# Phase P2 — Resume Parser mock verification

Install CPU-only dependencies, run the test suite, and exercise the API with synthetic data. This notebook does not download model weights or use private resumes.

In [ ]:
from pathlib import Path
import os

candidates = [Path('/content/ATS-CV/PDFPARSER'), Path.cwd(), Path.cwd() / 'PDFPARSER']
project_root = next((path for path in candidates if (path / 'requirements-dev.txt').is_file()), None)
if project_root is None:
    raise RuntimeError('Clone the repository, then run this notebook from the PDFPARSER project')
os.chdir(project_root)
print(project_root)

In [ ]:
%pip install -q -r requirements-dev.txt

In [ ]:
!python -m pytest -q

In [ ]:
from fastapi.testclient import TestClient
from app.core.config import Settings
from app.main import create_app

with TestClient(create_app(Settings(_env_file=None))) as client:
    health = client.get('/api/v1/health')
    ready = client.get('/api/v1/ready')
    parsed = client.post('/api/v1/resumes/parse-text', json={'text': 'Kỹ năng: Python'})

assert health.status_code == 200
assert ready.status_code == 200
assert parsed.status_code == 200
assert parsed.json()['data']['skills'] == ['Python']
parsed.json()

In [ ]:
from tests.fixtures.documents import make_docx_bytes

with TestClient(create_app(Settings(_env_file=None))) as client:
    uploaded = client.post(
        '/api/v1/resumes/parse',
        files={'file': ('synthetic.docx', make_docx_bytes(), 'application/vnd.openxmlformats-officedocument.wordprocessingml.document')},
    )

assert uploaded.status_code == 200, uploaded.text
uploaded.json()